In [1]:
import scanpy as sc

In [3]:
ad_sc = sc.read_loom('/Users/christoffer/Downloads/l5_all_rev1.loom')

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
/Users/christoffer/miniconda3/envs/sc_py312/lib/python3.12/site-packages/anndata/_io/read.py:151: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  axis_df[k] = v
/Users/christoffer/miniconda3/envs/sc_py312/lib/python3.12/site-packages/anndata/_io/read.py:151: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  axis_df[k] = v
/Users/christoffer/miniconda3/envs/sc_py312/lib/python3.12/site-packages/anndata/_io/read.py:151: PerformanceWarning: D

In [8]:
ad_sc.obs.Age = ad_sc.obs.Age.astype(str)

In [24]:
ad_sc.obs[['Class','Age']]

,Class,Age
CellID,,
10X82_2:TCTCTCACCAGTTA,Neurons,"p21, p23"
10X82_2:TATTATCTACCAGA,Neurons,"p21, p23"
10X82_2:TATCCCAGATGGCA,Neurons,"p21, p23"
10X82_2:ATTACGTATGAATG,Neurons,"p21, p23"
10X82_2:ATACGTCAATAAGG,Neurons,"p21, p23"
...,...,...
10X43_2:GGTACAACAGTCGT,Neurons,p20
10X43_2:TAATGATGGGTTAC,Neurons,p20
10X43_2:CTGCAGCTTAGAGA,Neurons,p20


In [27]:
import re
import numpy as np
import pandas as pd

def _tokens(s):
    # split on commas, trim, lowercase, collapse spaces
    return [re.sub(r"\s+", " ", t.strip().lower()) for t in str(s).split(",") if t.strip()]

# canonical mapping (edit as you like)
CANON = {
    "cortex": "Cortex",
    "hippocampus": "Hippocampus",
    "dentate gyrus": "Hippocampus",
    "amygdala": "Amygdala",
    "olfactory bulb": "Olfactory bulb",
    "thalamus": "Thalamus",
    "hypothalamus": "Hypothalamus",
    "pallidum": "Pallidum",
    "striatum dorsal": "Striatum",
    "striatum ventral": "Striatum",
    "striatum dorsal, striatum ventral": "Striatum",
    "striatum dorsal,striatum ventral": "Striatum",
    "midbrain dorsal": "Midbrain",
    "midbrain ventral": "Midbrain",
    "midbrain dorsal,midbrain ventral": "Midbrain",
    "pons": "Pons",
    "medulla": "Medulla",
    "cerebellum": "Cerebellum",
    "pons,medullae,cerebellum": "Hindbrain",
    "subcommissural organ": "Subcommissural organ",
    "telencephalon": "Telencephalon",
    "brain": "Brain",
    "cns": "Brain",
    "spinal cord": "Spinal cord",
    "dorsal root ganglion": "DRG",
    "sympathetic ganglion": "Sympathetic ganglion",
    "dorsal root ganglion,sympathetic ganglion": "PNS",
    "enteric nervous system": "Enteric NS",
}

# priority order if a row mentions several regions
PRIORITY = [
    "Cortex","Hippocampus","Striatum","Thalamus","Hypothalamus","Cerebellum",
    "Midbrain","Pons","Medulla","Olfactory bulb","Amygdala","Pallidum",
    "Telencephalon","Subcommissural organ","Brain","Hindbrain","Spinal cord",
    "DRG","Sympathetic ganglion","PNS","Enteric NS"
]

def assign_macro_region(s):
    toks = _tokens(s)
    hits = []
    for t in toks:
        if t in CANON:
            hits.append(CANON[t])
        else:
            # try partials like "striatum dorsal, Striatum ventral, Dentate gyrus"
            if "striatum" in t: hits.append("Striatum")
            elif "midbrain" in t: hits.append("Midbrain")
            elif "dentate" in t: hits.append("Hippocampus")
            elif "medull" in t: hits.append("Medulla")
            elif "pons" in t: hits.append("Pons")
            elif "cerebell" in t: hits.append("Cerebellum")
    hits = list(dict.fromkeys(hits))  # unique, keep order
    if not hits:
        return np.nan
    # pick highest priority if multiple
    for p in PRIORITY:
        if p in hits:
            return p
    return hits[0]

In [29]:
region_col = "Region"  # or whatever holds those strings
ad_sc.obs["region_macro"] = ad_sc.obs[region_col].map(assign_macro_region)
ad_sc.obs["region_macro"].value_counts(dropna=False)

/var/folders/pm/2253tbm54v36j0l7lhshq73r0000gn/T/ipykernel_19047/574362140.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ad_sc.obs["region_macro"] = ad_sc.obs[region_col].map(assign_macro_region)


region_macro
Brain                   51448
Cortex                  22758
Thalamus                12717
Enteric NS              11640
Olfactory bulb          11264
Hippocampus             10706
Striatum                 7907
Telencephalon            6947
Midbrain                 6933
Cerebellum               6333
DRG                      2311
Spinal cord              2226
Hypothalamus             2033
Medulla                  1573
Pons                     1207
Amygdala                  958
Sympathetic ganglion      886
Pallidum                  838
Subcommissural organ      111
Name: count, dtype: int64

In [31]:
# example: keep central brain areas; drop PNS / spinal
KEEP = {"Cortex","Hippocampus","Striatum","Thalamus","Hypothalamus",
        "Cerebellum","Midbrain","Pons","Medulla","Olfactory bulb","Amygdala","Pallidum"}

ad_filt = ad_sc[ad_sc.obs["region_macro"].isin(KEEP)].copy()
print(f"Kept {ad_filt.n_obs:,} / {ad_sc.n_obs:,} spots")

Kept 85,227 / 160,796 spots


In [44]:
ad_sc.obs.Class.unique()

array([np.str_('Neurons'), np.str_('PeripheralGlia'), np.str_('Vascular'),
       np.str_('Oligos'), np.str_('Astrocytes'), np.str_('Ependymal'),
       np.str_('Immune')], dtype=object)

In [45]:
ad_sc

AnnData object with n_obs × n_vars = 160796 × 27998
    obs: 'Age', 'AnalysisPool', 'AnalysisProject', 'Bucket', 'CellConc', 'Cell_Conc', 'ChipID', 'Class', 'ClassProbability_Astrocyte', 'ClassProbability_Astrocyte,Immune', 'ClassProbability_Astrocyte,Neurons', 'ClassProbability_Astrocyte,Oligos', 'ClassProbability_Astrocyte,Vascular', 'ClassProbability_Bergmann-glia', 'ClassProbability_Blood', 'ClassProbability_Blood,Vascular', 'ClassProbability_Enteric-glia', 'ClassProbability_Enteric-glia,Cycling', 'ClassProbability_Ependymal', 'ClassProbability_Ex-Neurons', 'ClassProbability_Ex-Vascular', 'ClassProbability_Immune', 'ClassProbability_Immune,Neurons', 'ClassProbability_Immune,Oligos', 'ClassProbability_Neurons', 'ClassProbability_Neurons,Cycling', 'ClassProbability_Neurons,Oligos', 'ClassProbability_Neurons,Satellite-glia', 'ClassProbability_Neurons,Vascular', 'ClassProbability_OEC', 'ClassProbability_Oligos', 'ClassProbability_Oligos,Cycling', 'ClassProbability_Oligos,Vascular', 'Cl

In [46]:
ad_sc.obs = ad_sc.obs[['Class','region_macro']]
ad_filt.obs = ad_filt.obs[['Class','region_macro']]

In [47]:
ad_filt.write('../data/linnarsson_adolescence.h5ad')
ad_sc.write('../data/linnarsson_adolescence_full.h5ad')

In [14]:
list(ad_sc.obs.columns)

['Age',
 'AnalysisPool',
 'AnalysisProject',
 'Bucket',
 'CellConc',
 'Cell_Conc',
 'ChipID',
 'Class',
 'ClassProbability_Astrocyte',
 'ClassProbability_Astrocyte,Immune',
 'ClassProbability_Astrocyte,Neurons',
 'ClassProbability_Astrocyte,Oligos',
 'ClassProbability_Astrocyte,Vascular',
 'ClassProbability_Bergmann-glia',
 'ClassProbability_Blood',
 'ClassProbability_Blood,Vascular',
 'ClassProbability_Enteric-glia',
 'ClassProbability_Enteric-glia,Cycling',
 'ClassProbability_Ependymal',
 'ClassProbability_Ex-Neurons',
 'ClassProbability_Ex-Vascular',
 'ClassProbability_Immune',
 'ClassProbability_Immune,Neurons',
 'ClassProbability_Immune,Oligos',
 'ClassProbability_Neurons',
 'ClassProbability_Neurons,Cycling',
 'ClassProbability_Neurons,Oligos',
 'ClassProbability_Neurons,Satellite-glia',
 'ClassProbability_Neurons,Vascular',
 'ClassProbability_OEC',
 'ClassProbability_Oligos',
 'ClassProbability_Oligos,Cycling',
 'ClassProbability_Oligos,Vascular',
 'ClassProbability_Satellite-gl